# Modelling Quantum Fluids with the Gross-Pitaevskii Equation

This notebook runs a 2D **Gross-Pitaevskii Equation (GPE)** simulator (`GrossPitaevskiiEquation2D`). The GPE is the mean-field equation for dilute-gas Bose-Einstein Condensates (BECs), describing experiments with ultracold alkali atoms (e.g. rubidium, sodium) confined in optical or magnetic traps.

- **State variables**: `density` ($|\psi|^2$), `real` ($\Re(\psi)$), and `imag` ($\Im(\psi)$).
- **Key physical parameters**: `g` (interaction strength), `Omega` (rotation frequency), obstacle and trap geometry.
- **Dynamics**: Split-Step Fourier Method alternating between position-space (potential + interaction) and momentum-space (kinetic) half-steps.
- **Boundary conditions**: Naturally periodic via the FFT, with localized confinement provided by either a harmonic trap or a Woods-Saxon box trap.

### Output representation

In BEC experiments, **density** ($|\psi|^2$) is the directly measured observable (via absorption imaging). For ML surrogate models, we output the wavefunction as **(real, imag)** components rather than (density, phase), because the phase $\phi = \text{atan2}(\Im\psi, \Re\psi)$ has $2\pi$ branch cuts that make it discontinuous and hard for neural networks to learn. The real/imag representation is smooth everywhere and density is trivially recoverable as $\Re(\psi)^2 + \Im(\psi)^2$.

### What this notebook demonstrates

Three physically distinct regimes of increasing complexity:
1. **Linear quantum mechanics** — a single wavepacket in a harmonic trap ($g=0$, no vortices).
2. **Obstacle-driven vortex shedding** — a laser beam sweeps through a box-trapped BEC, producing von Kármán-like vortex streets.
3. **Rotation-driven vortex lattice** — sustained trap rotation nucleates an ordered Abrikosov triangular lattice.

## Governing equations

The time-dependent Gross-Pitaevskii Equation describes the dynamics of the condensate wavefunction $\psi(x,y,t)$:

$$
i \hbar \frac{\partial \psi}{\partial t} = \left( -\frac{\hbar^2}{2m}\nabla^2 + V(x,y,t) + g |\psi|^2 - \Omega L_z \right) \psi
$$

Where:

- $-\frac{\hbar^2}{2m}\nabla^2$ is the kinetic energy operator.
- $V(x,y,t)$ is the external trapping potential (harmonic trap, uniform optical Woods-Saxon box, obstacle, and/or disorder).
- $g|\psi|^2$ represents the non-linear mean-field interaction (repulsive for $g>0$, as in typical alkali BECs).
- $\Omega L_z$ is the rotating-frame energy term, where $\Omega$ is the frame angular velocity and $L_z = -i\hbar(x\partial_y - y\partial_x)$ is the $z$-component of angular momentum. Setting $\Omega > 0$ nucleates quantised vortices in the condensate.

In this simulator, we use dimensionless units where $\hbar = 1$ and $m = 1$. The wavefunction is stepped forward in time using the **Split-Step Fourier Method**, which is symplectic and exactly preserves the $L^2$ norm (total particle number) — a key advantage over generic ODE solvers for long-time quantum evolution.

To prepare physically realistic initial states, the simulator supports **imaginary-time evolution** ($t \to -i\tau$), which projects onto the lowest-energy state. This mimics the role of evaporative cooling in real experiments, ensuring the simulation starts from a proper ground state rather than an arbitrary wavepacket.

### Simulation Parameters Explained

The examples below configure parameters that control physically meaningful regimes:

- **`g`**: Non-linear interaction strength. $g>0$ models repulsive bosons; $g=0$ recovers the linear Schrödinger limit.
- **`Omega`**: Rotating-frame angular velocity ($-\Omega L_z$ term). Nucleates vortices above a critical frequency.
- **`wx`, `wy`**: Harmonic trap frequencies. Set to 0.0 when using a uniform Box Trap.
- **`box_type`, `box_param`, `ws_a`, `ws_V0`**: Woods-Saxon box trap parameters governing wall position, steepness, and height.
- **`box_anisotropy`**: Elliptical deformation of the box trap; acts as a stirring paddle when combined with rotation.
- **`spoon_strength`, `spoon_speed`, `spoon_width`, `spoon_radius`**: Moving laser obstacle parameters (amplitude, oscillation frequency, size, sweep radius).
- **`spoon_type`**: Obstacle trajectory — `"orbit"` (circular) or `"linear"` (sinusoidal back-and-forth).
- **`x0`, `y0`**: Initial condensate center.
- **`kx0`, `ky0`**: Initial momentum kick.
- **`width`**: Initial Gaussian width (smaller values induce stronger breathing modes).
- **`imaginary_time_steps`**: Number of imaginary-time cooling steps in Phase 1, used to converge to the physical ground state before real-time evolution. This is the numerical analogue of evaporative cooling in real experiments.
- **`initial_noise`**: Amplitude of random vacuum fluctuations added before cooling, needed to break symmetry for vortex nucleation under rotation.

### Physical modeling scope

This simulator is a mean-field $T\approx 0$ model (Gross-Pitaevskii Equation). External laser/disorder effects are represented as classical potentials in $V(x,y,t)$. It does **not** explicitly model decoherence, thermal clouds, or stochastic dissipation; those require extensions such as SGPE/ZNG-type models.

In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import GrossPitaevskiiEquation2D as GPESim
from autosim.utils import plot_spatiotemporal_video


### Low Complexity Example

When $g=0$, the system reduces to the linear Schrödinger equation. With a perfectly round harmonic trap (`wx=1.0, wy=1.0`), the dynamics are exactly solvable.

If we place a Gaussian wavepacket inside this trap, its behaviour depends heavily on its initial `width`:

- **Coherent state (`width=1.0`)**: If the initial width matches the trap's natural length scale ($a_{ho} = 1/\sqrt{\omega_x} = 1.0$), the wavepacket is an eigenstate of the ladder operators. It oscillates rigidly back and forth without changing shape — exactly like a classical ball on a spring (Ehrenfest's theorem for the harmonic oscillator).
- **Squeezed state (`width=0.4`)**: When the initial width is narrower than $a_{ho}$, the wavepacket is a superposition of many energy eigenstates. These evolve at different frequencies, causing the wavepacket to periodically spread and refocus as the eigenstates dephase and rephase.

**What you will see:**

1. **Sloshing:** The wavepacket center oscillates in the trap at the trap frequency $\omega$.
2. **Breathing:** The width periodically expands and contracts at $2\omega$ as the squeezed eigenstates dephase.
3. **Interference fringes:** As the spreading wavepacket overlaps with itself during refocusing, constructive and destructive interference produces high-contrast density ripples.

In [ ]:
sim_easy = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,  # grid resolution 128x128
    T=24.0,  # 24/3.14 ≈ 7.64 breathing periods, ~4 full periods (T_trap = 2π ≈ 6.28)
    dt=0.005,
    snapshot_dt=0.1,  # finer snapshots to capture the breathing clearly
    L=10.0,
    parameters_range={
        "g": (0.0, 0.0),
        "disorder_strength": (0.0, 0.0),
        "spoon_strength": (0.0, 0.0),
        "wx": (1.0, 1.0),
        "wy": (1.0, 1.0),
        # Matplotlib imshow displays the mathematical Y-axis vertically by default.
        # Simulation creates grids using ij indexing, meaning X is row (vertical).
        # Push the packet along the y-axis making it slosh purely horizontally visually.
        "x0": (0.0, 0.0),
        "y0": (1.5, 1.5),
        "kx0": (0.0, 0.0),
        "ky0": (-2.0, -2.0),
        # width < a_ho = 1/sqrt(wx) = 1.0 => squeezed state: packet breathes and
        # spreads, producing visible interference fringes as it overlaps with itself
        "width": (0.4, 0.4),
    },
    random_seed=42,
)

res_easy = sim_easy.forward_samples_spatiotemporal(n=1)

print("Constant Scalars (Batch x Params):", res_easy["constant_scalars"].shape)
print(
    "Spatiotemporal Output:",
    res_easy["data"].shape,
    "[batch, time, x, y, channels={density, real, imag}]",
)


In [ ]:
anim_easy = plot_spatiotemporal_video(
    res_easy["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim_easy.to_jshtml())

### Vortex Shedding from a Moving Laser Obstacle

A focused laser beam obstacle oscillates back and forth through a stationary
BEC confined in a flat-bottomed optical box trap (Woods-Saxon profile). When
the obstacle speed exceeds the critical velocity $v_c$, the superfluid
response transitions from laminar potential flow to periodic
vortex-antivortex pair shedding (a quantum von Kármán street), and at
higher speeds to disordered turbulent wakes with sound emission.

This uses a **two-phase simulation**:
1. **Phase 1 (imaginary time):** converges to a clean, vortex-free ground state inside the uniform box trap.
2. **Phase 2 (real time):** the obstacle sweeps through the condensate, shedding vortices.

Parameters here match the stable `gpe/laser_only_wake` config. No rotation
($\Omega = 0$) — all dynamics come purely from the moving obstacle.

**References:**
- Sasaki, Suzuki & Saito, PRL 104, 150404 (2010) — GPE simulations predicting
  the Bénard–von Kármán vortex street in a 2D BEC with a Gaussian obstacle
  moving through a homogeneous background (closest theoretical match to this simulation).
- Reeves et al., PRL 114, 155302 (2015) — systematic GPE study of a Gaussian
  obstacle in a uniform 2D superfluid, defining the superfluid Reynolds number
  Re$_s$ and classifying shedding regimes (oblique dipoles → von Kármán street → turbulence).
- Kwon et al., PRL 117, 245301 (2016) — first experimental observation of a
  von Kármán vortex street in an atomic BEC ($^{23}$Na in a harmonic trap).

**What you will see:**

1. **Laminar-to-turbulent transition:** at turnaround (slow) the flow is laminar; at mid-sweep (fast) vortex pairs are shed.
2. **Von Kármán vortex street:** periodic alternating vortex-antivortex pairs trailing the obstacle.
3. **Sound emission:** density waves radiated when the obstacle speed exceeds the speed of sound $c_s = \sqrt{g \cdot n}$.

In [ ]:
sim_wake = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,
    L=10.0,
    T=17.55,
    dt=0.005,
    snapshot_dt=0.1,
    skip_nt=30,
    box_type="woods_saxon",
    spoon_type="linear",
    parameters_range={
        # Flat box trap (no harmonic component)
        "wx": (0.0, 0.0),
        "wy": (0.0, 0.0),
        "box_param": (0.002, 0.002),
        "ws_a": (0.15, 0.15),
        "ws_V0": (200.0, 200.0),
        "box_anisotropy": (1.0, 1.0),
        # Broad initial Gaussian fills box after cooling
        "width": (1.8, 1.8),
        "x0": (0.0, 0.0),
        "y0": (0.0, 0.0),
        "kx0": (0.0, 0.0),
        "ky0": (0.0, 0.0),
        # Phase 1: imaginary-time cooling to ground state
        "imaginary_time": (0, 0),
        "imaginary_time_steps": (2000, 2000),
        "imaginary_time_dt": (0.05, 0.05),
        "initial_noise": (0.0, 0.0),
        # Interaction strength (sets speed of sound)
        "g": (150.0, 150.0),
        # No disorder
        "disorder_strength": (0.0, 0.0),
        "disorder_radius": (1.0, 1.0),
        "disorder_time_dependent": (0, 0),
        # Moving obstacle (sinusoidal linear sweep)
        "spoon_strength": (5.0, 5.0),
        "spoon_speed": (1.2, 1.2),
        "spoon_width": (1.5, 1.5),
        "spoon_radius": (2.5, 2.5),
        # No rotation
        "Omega": (0.0, 0.0),
    },
    random_seed=42,
)

res_wake = sim_wake.forward_samples_spatiotemporal(n=1)
print("Output shape:", res_wake["data"].shape)

In [ ]:
anim_wake = plot_spatiotemporal_video(
    res_wake["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True,
)
HTML(anim_wake.to_jshtml())

### Rotating Box Vortex Lattice

A rotation-driven vortex lattice is a qualitatively different regime from the wake-shedding obstacle family.

Here we remove the spoon and disorder, use a pure uniform Woods-Saxon **circular box trap**, and apply sustained rotation so vortices organise into an ordered array rather than being continuously scrambled by obstacle forcing.

In an infinite uniform superfluid the lowest-energy vortex arrangement is an Abrikosov triangular (hexagonal) lattice. In a **circular box**, boundary effects impose concentric ring structures: vortices sit on rings of increasing radius rather than on a perfect triangular grid. With enough vortices the interior approximates the triangular lattice, but the edge geometry always reflects the circular symmetry of the trap.

This uses a **two-phase simulation**:
1. **Phase 1 (imaginary time):** 8000 steps converge to the rotating ground state — the vortex array is already present at the start of real-time evolution.
2. **Phase 2 (real time):** free evolution from the ground state. The vortex array co-rotates with the trap; residual noise from `initial_noise` can excite slow Tkachenko (lattice phonon) modes.

Parameters here match the stable `gpe/rotating_box_lattice` config.

**References:**
- Adhikari, J. Phys.: Condens. Matter 31, 275401 (2019) — GPE simulation of
  vortex lattices in a uniform BEC in a box trap using imaginary-time
  propagation with $\Omega L_z$ rotation. Uses the same numerical protocol
  as here; finds triangular-like interior ordering in a circular box and
  square ordering in a square box.
- Abo-Shaeer et al., Science 292, 476 (2001) — landmark experimental
  observation of ordered vortex lattices in a $^{23}$Na BEC. Note: that
  experiment used a harmonic magnetic trap with laser stirring, a different
  geometry from the uniform box trap simulated here.

**What you will see:**

1. **Ordered vortex array:** density zeros arranged on concentric rings, with approximately triangular ordering in the bulk.
2. **Co-rotation:** the vortex array rotates with the trap at $\approx \Omega$.
3. **Slow dynamics:** any residual Tkachenko-like distortions of the vortex positions evolving over many trap periods.

In [ ]:
sim_lattice = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,
    L=10.0,
    T=16.05,
    dt=0.005,
    snapshot_dt=0.1,
    box_type="woods_saxon",
    parameters_range={
        # Flat box trap (no harmonic component)
        "wx": (0.0, 0.0),
        "wy": (0.0, 0.0),
        "box_param": (0.0039, 0.0039),
        "ws_a": (0.15, 0.15),
        "ws_V0": (200.0, 200.0),
        "box_anisotropy": (1.05, 1.05),
        # Initial wavefunction
        "width": (1.0, 1.0),
        "x0": (0.0, 0.0),
        "y0": (0.0, 0.0),
        "kx0": (0.0, 0.0),
        "ky0": (0.0, 0.0),
        # Phase 1: long imaginary-time cooling for lattice ordering
        "imaginary_time": (0, 0),
        "imaginary_time_steps": (8000, 8000),
        "imaginary_time_dt": (0.05, 0.05),
        "initial_noise": (0.05, 0.05),
        # Strong repulsion prevents centrifugal evacuation
        "g": (400.0, 400.0),
        # No disorder or obstacle
        "disorder_strength": (0.0, 0.0),
        "disorder_radius": (1.0, 1.0),
        "disorder_time_dependent": (0, 0),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "spoon_width": (0.18, 0.18),
        "spoon_radius": (1.5, 1.5),
        # Rotation nucleates the lattice
        "Omega": (0.85, 0.85),
    },
    random_seed=42,
)

res_lattice = sim_lattice.forward_samples_spatiotemporal(n=1)
print("Output shape:", res_lattice["data"].shape)

In [ ]:
anim_lattice = plot_spatiotemporal_video(
    res_lattice["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True,
)
HTML(anim_lattice.to_jshtml())